# Stackformer: SegFormer Semantic Segmentation Training

This tutorial demonstrates configuring and training SegFormer on 2D semantic image segmentation using `Trainer`.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from stackformer.models.vision import SegFormer
from stackformer.training import TrainConfig, Trainer

# 1. Model Definition: SegFormer-B0 architecture
model = SegFormer(variant="b0", num_labels=4)
print(f"SegFormer parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Prepare Synthetic Image & Mask Dataset

In [ ]:
torch.manual_seed(42)
# Input images: (B, 3, 64, 64)
train_x = torch.randn(16, 3, 64, 64)
# Ground-truth segmentation masks: (B, 64, 64)
train_y = torch.randint(0, 4, (16, 64, 64))
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=4, shuffle=True)

val_x = torch.randn(8, 3, 64, 64)
val_y = torch.randint(0, 4, (8, 64, 64))
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=4)

## 3. Train with Stackformer Trainer

The built-in `loss="seg_cross_entropy"` automatically aligns decode head spatial dimensions with the target mask.

In [ ]:
cfg = TrainConfig(
    max_epochs=2,
    lr=1e-3,
    loss="seg_cross_entropy",
    checkpoint_dir="./checkpoints_segformer",
    eval_every_n_epochs=1,
    display="auto",
)

trainer = Trainer(model, cfg, train_loader=train_loader, eval_loader=val_loader)
state = trainer.train()
print(f"SegFormer training completed! Global step: {state.global_step}")